# Ultra-Diffuse Galaxy Jeans Modelling — Walkthrough

This notebook is an interactive companion to the Python implementation in `src/`. It demonstrates the adopted stellar density models, LOS dispersions, and DF44 NFW fits.

The published observational data are in `data/`. The full reproducible command is `python -m src.run_analysis`.

**Caution:** The DF2/DF4 GC points are individual velocities, not resolved dispersion bins. The DF44 fits are conditional on spherical, equilibrium, constant-anisotropy models.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np
from IPython.display import Image, display

ROOT = Path.cwd()
if not (ROOT / 'src').is_dir():
    ROOT = ROOT.parent
if not (ROOT / 'src').is_dir():
    raise RuntimeError('Start the notebook from the repository root or notebooks/ directory.')
sys.path.insert(0, str(ROOT))
from src.udg_jeans import (JeansGrid, make_tracer, df2_df4_density,
    df44_stellar_tracer, los_dispersion, df44_fit, load_data)
print('Repository:', ROOT)

## 1. Observational inputs

The DF44 table contains **nine binned stellar velocity dispersions**. The DF2/DF4 tables contain **individual globular-cluster velocities**.

In [ ]:
df2, df4, df44 = load_data(ROOT / 'data')
print('DF2 GC count:', len(df2), 'DF4 GC count:', len(df4), 'DF44 bins:', len(df44))
display(df44.head())

## 2. Stellar-only models

For DF2/DF4, the adopted analytic stellar profile is used as both a gravitating density and an approximate tracer distribution. DF44 adopts a deprojected Sérsic approximation with a constant stellar mass-to-light ratio.

In [ ]:
grid = JeansGrid.standard()
model_df2 = make_tracer(grid, df2_df4_density(grid.r, 1.25e7, 2.0))
model_df4 = make_tracer(grid, df2_df4_density(grid.r, 1.15e7, 1.0))
model_df44 = make_tracer(grid, df44_stellar_tracer(grid.r), mass_to_light=1.0)
for label, obj, beta in [('DF2',model_df2,0),('DF4',model_df4,0),('DF44 isotropic',model_df44,0)]:
    sig = los_dispersion([1.0], obj, obj.stellar_mass, beta)[0]
    print(f'{label:15s}: M_stars={obj.stellar_mass[-1]:.5e} Msun, sigma_LOS(1kpc)={sig:.3f} km/s')

## 3. Independent DF44 NFW fits

The baseline fixes concentration $c=10$. The second fit frees $c$ over the imposed numerical range $[2,40]$. Errors are asymmetric and are handled by a one-sided weighted residual. The fitted $c=2$ lies on a boundary: it is **not** an observational determination of concentration.

In [ ]:
fits = pd.DataFrame([
    df44_fit(df44, model_df44, free_concentration=False),
    df44_fit(df44, model_df44, free_concentration=True),
])
display(fits[['fit','m200_msun','c','beta','chi2','chi2_reduced_nominal','mhalo_at_6p3_kpc_msun']])

## 4. Generate the report figures and CSV results

The main script recomputes both fitted models from the data and generates five report figures.

In [ ]:
from src.run_analysis import main
main()

In [ ]:
display(Image(filename=str(ROOT / 'figures' / 'df44_nfw_bestfit.png')))
display(Image(filename=str(ROOT / 'figures' / 'df44_enclosed_halo_mass.png')))

## Interpretation and limitations

- The tested DF44 stellar-only models predict much smaller LOS dispersions than those observed, under Newtonian gravity.
- Both fitted NFW models have about $4.2\times 10^9 M_\odot$ of **halo** mass within 6.3 kpc despite very different extrapolated $M_{200}$.
- The DF2/DF4 GC comparison is qualitative because the observed points are **individual velocity offsets**, and GC density is not directly measured by the adopted stellar tracer model.
- The $\beta=+0.6$ DF44 illustration is a numerical Jeans solution but not globally physically admissible for the adopted central tracer cusp.
- No cosmological concentration prior, rigorous posterior intervals or observational aperture convolution is included.

See the [main README](../README.md) and [BTP report](../report/BTP_Report.pdf) for context and citations.